# Saluki for synonymous eGFP CDS design

[Notebook](saluki-egfp.ipynb) · [Installation](../docs/installation.md)

Score a single-exon eGFP transcript with fixed UTRs. Optimize the CDS with synonymous substitutions and retain the terminal stop codon. Install MEME Suite with the notebook dependencies. The first model load downloads the pretrained checkpoint.

## 1. Load the model



In [1]:
from pathlib import Path
from tempfile import mkdtemp

from MotifSift import SynonymousCodonSpace, call_motifs, select_candidates
from saluki_support import ReporterScorer, codon_ism
from reproduce import recorded_designs, recorded_device

device = recorded_device(exact_arithmetic=True)
cds = "".join(Path("data/egfp.fasta").read_text().splitlines()[1:])
model = ReporterScorer(len(cds), device=device)
seeds = [cds]
print("Starting Saluki score:", float(model.score(seeds)[0, 0]))

run = Path(mkdtemp(prefix="motifsift-example-"))


Starting Saluki score: 1.015636682510376




## 2. Generate candidates

Run the eleven optimizers and keep 100 candidates per optimizer. The displayed tables come from the full run reproduced by `examples/reproduce.py`.



In [2]:
designs = recorded_designs(
    "saluki", model, length=len(cds), initial=seeds, objective="maximize",
    gain_fractions=(0.5, 0.9), output_dir=run / "designs",
    is_CDS_optimized=True, fixed_region=[slice(len(cds) - 3, len(cds))],
)
designs.groupby(["algorithm", "gain_fraction"])["score"].agg(["size", "median", "max"])


size    median       max
algorithm                       gain_fraction                          
adalead                         0.5             100  1.516985  1.681509
                                0.9             100  2.588798  2.645017
basesearch                      0.5             100  1.433591  1.508610
                                0.9             100  2.516793  2.552695
beam_search                     0.5             100  1.562780  1.689251
                                0.9             100  2.678402  2.724082
cross_entropy_method            0.5             100  1.453089  1.590080
                                0.9             100  2.483534  2.522873
den                             0.5             100  1.396423  1.745093
                                0.9             100  2.107789  2.196756
fast_seqprop                    0.5             100  1.283291  1.754693
                                0.9             100  2.252657  2.447331
genetic_algorithm               0.5             100  1.565318  1.828779
                                0.9             100  2.651065  2.722817
ism_greedy                      0.5             100  1.379088  1.530929
                                0.9             100  2.370621  2.487951
ledidi                          0.5             100  0.787613  1.078837
                                0.9             100  1.319844  1.483486
simulated_annealing             0.5             100  0.591725  1.109233
                                0.9             100  0.913594  1.109233
vanilla_activation_maximization 0.5             100  1.501908  1.996902
                                0.9             100  2.235355  2.428951



## 3. Discover motifs

Analyze the 90% gain stage with synonymous-codon attribution and forward-strand motif discovery.



In [3]:
designs = designs.loc[designs["gain_fraction"].eq(0.9)].copy()
space = SynonymousCodonSpace(cds).with_fixed_positions(cds, range(len(cds) - 3, len(cds)))
contributions = codon_ism(model, designs.drop_duplicates("sequence"), space)
motifs = call_motifs(
    designs, reverse_complement=False, contributions=contributions,
    output_dir=run / "motifs",
)
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
1,2-UCACGCUUAA,643,643,930.0
2,3-GUACCACUGGM,475,475,975.0
4,6-CGGAAAGC,515,510,1315.0
5,7-ACAAUUACAAU,10,10,2538.0
6,8-GGCGAG,58,58,1923.0
7,10-UUAAUGGGCA,11,11,553.0
8,9-CUACGCUCAC,495,477,314.0
11,15-GCAGCGUGCA,12,12,601.0
15,23-UGAGCACCCA,431,431,858.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,ATGGTTAGTAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCC...,2.724082,2-UCACGCUUAA;3-GUACCACUGGM;6-CGGAAAGC;7-ACAAUU...
664,2,ATGGTGAGTAAGGGGGAGGAGTTGTTTACGGGGGTGGTTCCGATCT...,2.265943,7-ACAAUUACAAU;10-UUAAUGGGCA
789,3,ATGGTTTCGAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCGATCT...,2.184092,7-ACAAUUACAAU;8-GGCGAG;15-GCAGCGUGCA
662,4,ATGGTTTCGAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCCATCC...,2.267606,2-UCACGCUUAA;6-CGGAAAGC;7-ACAAUUACAAU
446,5,ATGGTTAGTAAGGGGGAGGAGTTGTTCACGGGGGTGGTGCCGATCC...,2.484123,2-UCACGCUUAA;3-GUACCACUGGM;6-CGGAAAGC;7-ACAAUU...
656,6,ATGGTTAGTAAGGGGGAGGAGTTGTTCACTGGGGTGGTGCCGATCT...,2.277158,6-CGGAAAGC;7-ACAAUUACAAU;8-GGCGAG;10-UUAAUGGGC...
555,7,ATGGTCTCGAAGGGGGAGGAGTTGTTCACCGGGGTCGTGCCCATCC...,2.377772,2-UCACGCUUAA;3-GUACCACUGGM;6-CGGAAAGC;7-ACAAUU...
634,8,ATGGTTAGCAAGGGGGAGGAGTTGTTTACGGGGGTGGTTCCGATCC...,2.305607,3-GUACCACUGGM;7-ACAAUUACAAU;8-GGCGAG;15-GCAGCG...
799,9,ATGGTTTCGAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCT...,2.171929,7-ACAAUUACAAU;8-GGCGAG;10-UUAAUGGGCA
742,10,ATGGTTTCGAAGGGGGAGGAGCTCTTCACCGGCGTCGTGCCCATCC...,2.229395,2-UCACGCUUAA;6-CGGAAAGC;7-ACAAUUACAAU;8-GGCGAG...
